# DiaRisk — Step 1: Basic Data Analysis

Educational EDA on the **Pima Indians Diabetes** dataset.

> **Disclaimer:** Demo only — not medical advice.

**Goal of this notebook:** understand the data before we train a model.

## 1. Imports and load data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw" / "pima-indians-diabetes.csv"

COLUMNS = [
    "pregnancies",
    "glucose",
    "blood_pressure",
    "skin_thickness",
    "insulin",
    "bmi",
    "diabetes_pedigree",
    "age",
    "outcome",
]

df = pd.read_csv(RAW, header=None, names=COLUMNS)
df.head()

## 2. Shape and dtypes

How many rows/columns? Are types numeric?

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print()
df.info()

## 3. Target balance (`outcome`)

- `0` = no diabetes
- `1` = diabetes

Imbalance matters later when we choose metrics (not only accuracy).

In [ ]:
counts = df["outcome"].value_counts().sort_index()
percents = df["outcome"].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({"count": counts, "percent": percents.round(1)})
summary.index = summary.index.map({0: "no diabetes (0)", 1: "diabetes (1)"})
display(summary)

ax = counts.plot(kind="bar", color=["#4C78A8", "#F58518"], rot=0)
ax.set_title("Target balance")
ax.set_xlabel("outcome")
ax.set_ylabel("count")
ax.set_xticklabels(["no diabetes", "diabetes"])
plt.tight_layout()
plt.show()

## 4. Descriptive statistics

Mean, std, min/max — first feel for the feature ranges.

In [ ]:
df.describe().T.round(2)

## 5. Zeros that may mean missing

In this dataset, `0` is often **not physiologically realistic** for some fields  
(e.g. glucose, BMI, blood pressure) and usually means **missing**.

We only **observe** this here. Cleaning comes in a later step.

In [ ]:
zero_cols = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
zero_counts = pd.Series({col: int((df[col] == 0).sum()) for col in zero_cols})
zero_counts = zero_counts.to_frame("n_zeros")
zero_counts["percent"] = (100 * zero_counts["n_zeros"] / len(df)).round(1)
zero_counts

## 6. Feature distributions

Histograms for each feature, colored by `outcome`.

In [ ]:
feature_cols = [c for c in df.columns if c != "outcome"]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for ax, col in zip(axes, feature_cols):
    sns.histplot(
        data=df,
        x=col,
        hue="outcome",
        bins=20,
        element="step",
        stat="density",
        common_norm=False,
        ax=ax,
        palette={0: "#4C78A8", 1: "#F58518"},
    )
    ax.set_title(col)
    ax.set_xlabel("")

fig.suptitle("Feature distributions by outcome", y=1.02)
plt.tight_layout()
plt.show()

## 7. Correlation heatmap

Which features move together? Which relate most to `outcome`?

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

print("Correlation with outcome (sorted):")
print(corr["outcome"].drop("outcome").sort_values(ascending=False).round(3))

## 8. Takeaways for the next step

1. **768 rows**, 8 features + binary target — small and good for learning.
2. Classes are **imbalanced** (~65% / 35%) → use precision/recall/ROC later, not only accuracy.
3. Some zeros are likely **missing values** → handle before/while training.
4. **Glucose** and **BMI** usually correlate most with diabetes outcome.

**Next (Step 2):** train a first classifier with train/test split.